# v0.3 — BloodMNIST (white blood cell) CNN inference on AUP-ZU3

End-to-end hardware inference of the v0.4 wider standard CNN on the AUP-ZU3
board (Zynq UltraScale+ XCZU3EG).

Pipeline that produced the bitstream loaded by this notebook:
1. `01.training` — Keras + QKeras CNN, 64x64x3 input, 8 classes, distilled, ~89% software accuracy.
2. `02.hls4ml` — hls4ml conversion to Vitis HLS C++, `ap_fixed<24,12>` precision (the network collapses at `<16,6>`).
3. `03.hls` — AXI-Stream + AXI-Lite wrapper exposing `s_axis_in`, `s_axis_out`, `s_axi_CTRL`. 24-bit value per beat.
4. `04.hw` — Vivado block design (PS + DMA + interconnect + our IP), bitstream + XSA. Measured: 157 DSP (44%), 50% LUT, 124 BRAM (57%), timing met.
5. `05.pynq` — **this notebook**.

Reference to beat: **software/bit-accurate accuracy is 93.25%** on the
3421-image 64x64 test set. The hardware should reproduce it within a
fraction of a point.



## 1. Imports and environment

Sanity check before touching the hardware.


In [ ]:
import time
import numpy as np
from pynq import Overlay, allocate
from utils64 import (
    image_to_axis_buffer,
    axis_buffer_to_floats,
    confusion_matrix_np,
    plot_confusion_matrix,
    WBC_CLASSES,
    WBC_PIXELS,
    FX_TOTAL_BITS,
    FX_FRAC_BITS,
)

print(f'numpy version: {np.__version__}')
print(f'Fixed-point: {FX_TOTAL_BITS} bits total, {FX_FRAC_BITS} fractional bits (ap_fixed<24,12>)')
print(f'Input size:  {WBC_PIXELS} values per image (64 x 64 x 3)')
print(f'Classes:     {len(WBC_CLASSES)} -> {WBC_CLASSES}')



## 2. Load the FPGA overlay

First critical step. If the cell returns silently (kernel does not die),
the PL was programmed successfully.


In [ ]:
ol = Overlay('bd_wrapper_wbc64_v05.xsa')
print('Overlay loaded.')



## 3. Discover IP cores in the overlay

Expected names:
- `axi_dma_0`
- `myproject_wbc64_v05_accel_0`
- `zynq_ultra_ps_e_0`

If the accelerator name differs, adjust `IP_NAME` in the next cell.



In [ ]:
list(ol.ip_dict.keys())


## 4. Create handles for DMA and the custom IP

- `sendchannel` — PS -> PL (the 64x64x3 image).
- `recvchannel` — PL -> PS (8 class scores).
- AXI-Lite CTRL register at offset 0x00: writing `1` triggers the IP.



In [ ]:
IP_NAME = 'myproject_wbc64_v05_accel_0'
IP_CTRL_REG = 0x00

dma = ol.axi_dma_0
dma_send = dma.sendchannel
dma_recv = dma.recvchannel

ip = getattr(ol, IP_NAME)
print(f'DMA handle: {dma}')
print(f'IP handle:  {ip}')



## 5. Allocate DMA-coherent buffers

- Input: 12288 AXI beats of 32 bits each — one per (pixel, channel) value
  of a 64x64x3 image. Each beat carries a 24-bit `fixed<24,12>` in its low
  24 bits (high 8 unused).
- Output: 8 beats, one per class score (also `fixed<24,12>`).

`allocate` returns physically contiguous memory visible to the AXI DMA.



In [ ]:
N_IN  = WBC_PIXELS  # 12288 = 48 * 48 * 3
N_OUT = len(WBC_CLASSES)  # 8

in_buffer  = allocate(shape=(N_IN,),  dtype=np.uint32)
out_buffer = allocate(shape=(N_OUT,), dtype=np.uint32)

print(f'in_buffer:  shape={in_buffer.shape},  dtype={in_buffer.dtype},  phys_addr=0x{in_buffer.physical_address:08x}')
print(f'out_buffer: shape={out_buffer.shape}, dtype={out_buffer.dtype}, phys_addr=0x{out_buffer.physical_address:08x}')



## 6. Load the BloodMNIST test set (already 64x64)

The dataset was native 64x64 (no resize needed) **on the host** with the exact same
`tf.image.resize` used during training, then saved as a compressed npz.
This avoids any resize on the board (PYNQ has no TensorFlow, and
`cv2`/`PIL` bilinear is not bit-identical to TF's). The board only packs
and streams.

`bloodmnist64_test.npz` keys: `x` (N, 48, 48, 3) float32 in [0,1], `y` (N,) int64 in 0..7.



In [ ]:
import os

NPZ_PATH = 'test_dataset/bloodmnist64_test.npz'
if not os.path.exists(NPZ_PATH):
    raise FileNotFoundError(
        f'{NPZ_PATH} not found. Generate it on the host with '
        f'tf.image.resize(x64, (48,48)) and SCP it next to this notebook.'
    )

data = np.load(NPZ_PATH)
images = data['x'].astype(np.float32)   # (N, 48, 48, 3)
labels = data['y'].astype(int)          # (N,)

N = images.shape[0]
print(f'Loaded {N} samples, shape {images.shape}, pixel range [{images.min():.3f}, {images.max():.3f}]')
from collections import Counter
print('Class distribution:', dict(sorted(Counter(labels.tolist()).items())))



## 7. Visual sanity check on a single image

Confirms the (H, W, C) layout is right and the pixels look like a real
cell image, not garbage.


In [ ]:
import matplotlib.pyplot as plt

idx = 0
img = images[idx]
label = labels[idx]

plt.figure(figsize=(2.4, 2.4))
plt.imshow(img)
plt.title(f'idx={idx}, label={label}\n({WBC_CLASSES[label]})')
plt.axis('off')
plt.show()


## 8. Run inference on a single image (step by step)

Verbose version. If any DMA `transfer`/`wait` hangs or the IP refuses to
raise `ap_done`, this is where it shows. The order of DMA operations is
the one validated on this board in v0.2:
`ip.write -> send.transfer -> recv.transfer -> send.wait -> recv.wait`.


In [ ]:
idx = 0
img = images[idx]
label = labels[idx]

print('1. Packing image into input buffer...')
t0 = time.time()
image_to_axis_buffer(img, in_buffer)
print(f'   done in {(time.time()-t0)*1000:.1f} ms')

ctrl = ip.read(0x00)
print(f'2. Pre-CTRL = 0x{ctrl:08x} (ap_idle={(ctrl>>2)&1})')

print('3. Trigger IP, push send (12288 beats), arm recv...')
ip.write(0x00, 1)
dma_send.transfer(in_buffer)
dma_recv.transfer(out_buffer)

print('4. Wait for send...')
t0 = time.time(); dma_send.wait(); print(f'   send done in {(time.time()-t0)*1000:.2f} ms')

print('5. Wait for recv...')
t0 = time.time(); dma_recv.wait(); print(f'   recv done in {(time.time()-t0)*1000:.2f} ms')

ctrl = ip.read(0x00)
print(f'6. Post-CTRL = 0x{ctrl:08x} (ap_done={(ctrl>>1)&1}, ap_idle={(ctrl>>2)&1})')

scores = axis_buffer_to_floats(out_buffer)
pred = int(np.argmax(scores))
print('\nRaw scores:')
for i, s in enumerate(scores):
    marker = ' <-- argmax' if i == pred else ''
    print(f'  class {i} ({WBC_CLASSES[i]:>20s}): {s:+.5f}{marker}')
print()
print(f'Predicted: {pred} ({WBC_CLASSES[pred]})')
print(f'Truth:     {label} ({WBC_CLASSES[label]})')
print('CORRECT' if pred == label else 'WRONG')



## 9. Inference helper

Wrap the validated step order into one function.


In [ ]:
def inference_image(img):
    image_to_axis_buffer(img, in_buffer)
    ip.write(0x00, 1)
    dma_send.transfer(in_buffer)
    dma_recv.transfer(out_buffer)
    dma_send.wait()
    dma_recv.wait()
    return axis_buffer_to_floats(out_buffer)


## 10. Smoke test: 10 images

Confirm the pipeline is stable and predictions are sane before the full
set. Expected: most of these correct (software accuracy is 83%).


In [ ]:
n_smoke = 10
correct = 0
for i in range(n_smoke):
    scores = inference_image(images[i])
    pred = int(np.argmax(scores))
    ok = pred == labels[i]
    correct += int(ok)
    print(f'  i={i:2d}  pred={pred} ({WBC_CLASSES[pred]:>20s})  true={labels[i]} ({WBC_CLASSES[labels[i]]:>20s})  {"ok" if ok else "WRONG"}')
print(f'\nSmoke-test accuracy: {correct}/{n_smoke}')


## 11. Full test set

Run inference over all 3421 test samples, measuring wall-clock latency.
Expected: ~89% accuracy (software/bit-accurate gave 93.25%). The wall-clock
figure is dominated by the per-image Python packing, not the hardware.



In [ ]:
correct = 0
preds = []

t0 = time.time()
for i in range(N):
    scores = inference_image(images[i])
    pred = int(np.argmax(scores))
    preds.append(pred)
    if pred == labels[i]:
        correct += 1
    if i % 500 == 0:
        elapsed = time.time() - t0
        print(f'  {i:5d}/{N}  elapsed={elapsed:6.1f}s  running acc={correct/max(i,1):.4f}')

total = time.time() - t0
preds = np.array(preds)

accuracy = correct / N
print()
print(f'Hardware accuracy on {N} samples: {accuracy:.4f}  ({correct}/{N})')
print(f'Total wall-clock time: {total:.1f} s')
print(f'Per-image average: {total/N*1000:.2f} ms (includes Python packing overhead)')
print(f'Throughput: {N/total:.1f} images/s')


## 12. Compare against the software reference (per-image)

`ref_pred_sw64_v05.npy` holds the QKeras software prediction for every test
image, in the same order. If the hardware is faithful, the two label
vectors should agree on almost every image — not just in aggregate
accuracy but image by image. Any large disagreement points to a packing
or fixed-point bug, not just a weaker model.

(Optional: only runs if you also SCP'd `ref_pred_sw64_v05.npy` to the board.)



In [ ]:
REF_PATH = 'test_dataset/ref_pred_sw64_v05.npy'
if os.path.exists(REF_PATH):
    ref = np.load(REF_PATH)
    agree = int((ref == preds).sum())
    print(f'HW vs SW agreement: {agree}/{N} = {agree/N*100:.2f}%')
    print(f'SW accuracy (ref): {(ref==labels).mean()*100:.2f}%')
    print(f'HW accuracy:       {(preds==labels).mean()*100:.2f}%')
    diff = np.where(ref != preds)[0]
    print(f'Images where HW and SW disagree: {len(diff)}')
    if len(diff):
        print('First few disagreements (idx, sw_pred, hw_pred, truth):')
        for k in diff[:10]:
            print(f'  {k:5d}  sw={ref[k]}  hw={preds[k]}  truth={labels[k]}')
else:
    print(f'{REF_PATH} not present — skipping per-image comparison.')



## 13. Confusion matrix


In [ ]:
cm = confusion_matrix_np(labels, preds, len(WBC_CLASSES))
plot_confusion_matrix(cm, class_names=WBC_CLASSES, normalize=False)


## 14. Pure-hardware latency (no Python overhead)

Pre-pack one image once, then measure only the DMA + IP execution loop.
This isolates the true hardware latency from the interpreted packing loop.


In [ ]:
image_to_axis_buffer(images[0], in_buffer)

# Warm up
for _ in range(20):
    ip.write(IP_CTRL_REG, 1)
    dma_send.transfer(in_buffer)
    dma_recv.transfer(out_buffer)
    dma_send.wait()
    dma_recv.wait()

n_meas = 2000
t0 = time.time()
for _ in range(n_meas):
    ip.write(IP_CTRL_REG, 1)
    dma_send.transfer(in_buffer)
    dma_recv.transfer(out_buffer)
    dma_send.wait()
    dma_recv.wait()
hw_total = time.time() - t0

print(f'Hardware-only loop ({n_meas} iters on the same pre-packed image):')
print(f'  total = {hw_total:.3f} s')
print(f'  per-image = {hw_total/n_meas*1e6:.1f} us')
print(f'  throughput = {n_meas/hw_total:,.0f} images/s')


----

### Appendix: how `bloodmnist64_test.npz` and `ref_pred_sw64_v05.npy` were made

Run **on the host** in `neuralEnv10` (the board has no TensorFlow):

```python
import numpy as np, tensorflow as tf
from qkeras.utils import _add_supported_quantized_objects
from tensorflow.keras.models import load_model

z = np.load('00.dataset/bloodmnist64_test.npz')
x48 = tf.image.resize(z['x'], (48, 48)).numpy().astype(np.float32)  # same as training
np.savez_compressed('00.dataset/bloodmnist64_test.npz', x=x48, y=z['y'].astype(np.int64))

co = {}; _add_supported_quantized_objects(co)
model = load_model('01.training/models/distilled_cnn48.h5', custom_objects=co)
y_sw = np.argmax(model.predict(x48, verbose=0), axis=1)
np.save('ref_pred_sw64_v05.npy', y_sw)   # software accuracy 93.25%
```

